# PEFT 파인튜닝 - Gemma 4 E4B-it

- 모델: `google/gemma-4-E4B-it`
- 아키텍처: `AutoModelForImageTextToText` (멀티모달)
- 채팅 포맷: `<start_of_turn>user/model`
- 요구 사항: `transformers >= 5.5.0`, L4 GPU (22GB)


## 1. 라이브러리 설치

In [1]:
!pip uninstall -y triton 2>/dev/null
!pip install -q --upgrade \
    transformers peft trl bitsandbytes \
    accelerate datasets einops sentencepiece \
    "protobuf<6.0.0"
!pip uninstall -y triton 2>/dev/null

import importlib
for pkg in ["transformers", "peft", "trl", "bitsandbytes", "accelerate"]:
    try:
        m = importlib.import_module(pkg)
        print(f"{pkg:<15} : {m.__version__}")
    except Exception as e:
        print(f"{pkg:<15} : ERROR - {e}")

print()
print("설치 완료 - 런타임을 재시작하세요 (Ctrl+M .)")
print("재시작 후 이 셀을 한 번 더 실행한 뒤 섹션 2 로 진행하세요.")


Found existing installation: triton 3.6.0
Uninstalling triton-3.6.0:
  Successfully uninstalled triton-3.6.0
Found existing installation: triton 3.6.0
Uninstalling triton-3.6.0:
  Successfully uninstalled triton-3.6.0
transformers    : 5.9.0


peft            : 0.19.1
trl             : 1.5.0
bitsandbytes    : 0.49.2
accelerate      : 1.13.0

설치 완료 - 런타임을 재시작하세요 (Ctrl+M .)
재시작 후 이 셀을 한 번 더 실행한 뒤 섹션 2 로 진행하세요.


## 2. 환경 확인 (재시작 후 여기서 시작)

In [2]:
import torch
assert torch.cuda.is_available(), (
    "GPU 를 찾을 수 없습니다.\n"
    "메뉴 -> 런타임 -> 런타임 유형 변경 -> L4 GPU -> 저장 후 재시작하세요."
)
print(f"GPU  : {torch.cuda.get_device_name(0)}")
print(f"VRAM : {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

import transformers, peft, trl, bitsandbytes as bnb
print(f"transformers : {transformers.__version__}")
print(f"peft         : {peft.__version__}")
print(f"trl          : {trl.__version__}")
print(f"bitsandbytes : {bnb.__version__}")

from transformers.models.auto.configuration_auto import CONFIG_MAPPING
assert "gemma4" in CONFIG_MAPPING, (
    "Gemma 4 를 인식하지 못합니다. 섹션 1 을 다시 실행 후 런타임을 재시작하세요."
)
print("Gemma 4 지원   : OK")

print("환경 확인 완료 - 이후 셀을 실행하세요.")


GPU  : NVIDIA A100-SXM4-80GB
VRAM : 79.3 GB
transformers : 5.9.0
peft         : 0.19.1
trl          : 1.5.0
bitsandbytes : 0.49.2
Gemma 4 지원   : OK
환경 확인 완료 - 이후 셀을 실행하세요.


## 3. Drive 마운트 및 HF 로그인

In [3]:
from google.colab import drive, userdata
import os

drive.mount("/content/drive")

try:
    hf_token = userdata.get("HF_TOKEN")
    os.environ["HF_TOKEN"] = hf_token
    from huggingface_hub import login
    login(token=hf_token, add_to_git_credential=False)
    print("HuggingFace 로그인 완료")
except Exception as e:
    print(f"[WARN] HF_TOKEN 없음: {e}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
[WARN] HF_TOKEN 없음: Secret HF_TOKEN does not exist.


## 4. 경로 및 설정

In [4]:
import os

CHUNK_JSON_PATH = "/content/drive/MyDrive/data/RFP_project/dataset/kh_v3.json"
EVAL_CSV_DIR    = "/content/drive/MyDrive/data/RFP_project/dataset/eval"
OUTPUT_DIR      = "/content/drive/MyDrive/data/RFP_project/peft_output/gemma4-E4B_v2"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# LoRA
LORA_R       = 16
LORA_ALPHA   = 32
LORA_DROPOUT = 0.05
LORA_TARGET_MODULES = [
    "q_proj", "k_proj", "v_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj",
]

# 학습
MAX_SEQ_LEN   = 2048
BATCH_SIZE    = 8
GRAD_ACCUM    = 8
LEARNING_RATE = 2e-4
NUM_EPOCHS    = 3
WARMUP_RATIO  = 0.05
SAVE_STEPS    = 100

# 데이터
CHUNK_SAMPLE_N    = 5000
CHUNK_MIN_LEN     = 100
TRAIN_SPLIT_RATIO = 0.9

print("경로 및 설정 완료")
print(f"출력 경로: {OUTPUT_DIR}")


경로 및 설정 완료
출력 경로: /content/drive/MyDrive/data/RFP_project/peft_output/gemma4-E4B_v2


## 5. 데이터 로드

In [5]:
import json, glob, ast, random
import pandas as pd
from collections import defaultdict
from datasets import Dataset

random.seed(42)

# 청크 로드
with open(CHUNK_JSON_PATH, encoding="utf-8") as f:
    chunks = json.load(f)
print(f"총 청크 수 : {len(chunks):,}")

chunk_by_src: dict[str, list] = defaultdict(list)
for c in chunks:
    src = c.get("metadata", {}).get("source_file", "")
    chunk_by_src[src].append(c)

# eval CSV 로드
def safe_parse(val):
    if pd.isna(val) or str(val).strip() in ("", "[]", "{}", "nan"):
        return [] if str(val).strip() in ("", "[]", "nan") else {}
    try:
        return json.loads(val)
    except Exception:
        try:
            return ast.literal_eval(val)
        except Exception:
            return val

csv_files = sorted(glob.glob(os.path.join(EVAL_CSV_DIR, "*.csv")))
eval_df = pd.concat([pd.read_csv(p, dtype=str) for p in csv_files], ignore_index=True)
for col in ["ground_truth_docs", "metadata_filter", "history"]:
    eval_df[col] = eval_df[col].apply(safe_parse)

print(f"eval CSV {len(csv_files)}개  |  레코드 {len(eval_df):,}개")
print(eval_df["type"].value_counts().sort_index().to_string())


총 청크 수 : 38,287
eval CSV 38개  |  레코드 1,100개
type
A    328
B    311
C    143
D    162
E    156


## 6. 학습 데이터 구성

In [6]:
SYSTEM_PROMPT = (
    "당신은 공공기관 및 기업의 제안요청서(RFP) 분석 전문가입니다. "
    "주어진 문서 내용을 바탕으로 정확하고 간결하게 답변하세요. "
    "문서에 없는 정보는 추측하지 말고 '해당 정보는 문서에서 확인되지 않습니다'라고 답하세요."
)
RESPONSE_MARKER = "<start_of_turn>model\n"


def is_valid_chunk(text: str) -> bool:
    return len(text.replace("[TABLE]", "").replace("[IMAGE]", "").strip()) >= CHUNK_MIN_LEN


def retrieve_context(ground_truth_docs: list, max_chunks: int = 3) -> str:
    contexts = []
    for fname in ground_truth_docs:
        stem = fname.replace(".hwp", "").replace(".pdf", "").strip()
        for src, doc_chunks in chunk_by_src.items():
            if stem in src.replace(".hwp","").replace(".pdf","").strip():
                for c in doc_chunks[:max_chunks]:
                    cleaned = c["text"].replace("[TABLE]","").replace("[IMAGE]","").strip()
                    if cleaned:
                        contexts.append(cleaned)
                break
    return "\n\n".join(contexts[:max_chunks])


def fmt(s: dict) -> dict:
    return {"text": (
        f"<start_of_turn>system\n{SYSTEM_PROMPT}<end_of_turn>\n"
        f"<start_of_turn>user\n{s['instruction']}<end_of_turn>\n"
        f"<start_of_turn>model\n{s['response']}<end_of_turn>"
    )}


def make_dataset(raw: list) -> Dataset:
    return Dataset.from_list([fmt(s) for s in raw])


# 샘플 생성
valid_chunks   = [c for c in chunks if is_valid_chunk(c["text"])]
sampled        = random.sample(valid_chunks, min(CHUNK_SAMPLE_N, len(valid_chunks)))

summary_samples = []
for c in sampled:
    meta    = c.get("metadata", {})
    parent  = c.get("parent_text", c["text"])
    answer  = parent.replace("[TABLE]","").replace("[IMAGE]","").strip()[:500]
    summary_samples.append({
        "instruction": (
            f"다음은 '{meta.get('agency','해당 기관')}'의 "
            f"'{meta.get('project_name','해당 사업')}' RFP 일부입니다.\n\n"
            f"[문서 내용]\n{c['text']}\n\n"
            "사업 목적, 예산, 기간, 주요 요구사항 등 핵심 정보를 요약해 주세요."
        ),
        "response": answer,
    })

qa_samples = []
for _, row in eval_df.iterrows():
    gt_docs  = row["ground_truth_docs"] if isinstance(row["ground_truth_docs"], list) else []
    history  = row["history"] if isinstance(row["history"], list) else []
    context  = retrieve_context(gt_docs)
    user_msg = (
        f"[참고 문서]\n{context}\n\n[질문]\n{row['question']}"
        if context else row["question"]
    )
    if row["type"] == "C" and history:
        hist = "\n".join(
            f"사용자: {h.get('user','')}\n어시스턴트: {h.get('assistant','')}"
            for h in history if isinstance(h, dict)
        )
        user_msg = f"[이전 대화]\n{hist}\n\n{user_msg}"
    qa_samples.append({"instruction": user_msg, "response": row["ground_truth_answer"]})

all_samples = summary_samples + qa_samples
random.shuffle(all_samples)
split_idx   = int(len(all_samples) * TRAIN_SPLIT_RATIO)
train_raw   = all_samples[:split_idx]
val_raw     = all_samples[split_idx:]

train_dataset = make_dataset(train_raw)
val_dataset   = make_dataset(val_raw)

print(f"전체 {len(all_samples):,}  (요약 {len(summary_samples)} + QA {len(qa_samples)})")
print(f"학습셋 {len(train_dataset):,}  |  검증셋 {len(val_dataset):,}")


# Gemma 4 는 mm_token_type_ids 필드가 없으면 forward 에서 오류 발생
# 데이터셋에 미리 추가해두면 콜레이터 없이도 동작한다.
def add_mm_token_type_ids(example):
    example["mm_token_type_ids"] = [0] * len(example["input_ids"])
    return example

# SFTTrainer 가 토크나이징을 완료한 뒤 이 함수가 적용되어야 하므로
# 여기서는 텍스트 샘플에 더미 필드만 추가해둔다.
# 실제 토큰 길이 기준 mm_token_type_ids 는 학습 루프에서 처리한다.
print("데이터셋 준비 완료")

전체 6,100  (요약 5000 + QA 1100)
학습셋 5,490  |  검증셋 610
데이터셋 준비 완료


## 7. 모델 로드 및 LoRA 적용

In [7]:
# 이전 실행에서 모델이 메모리에 남아있으면 OOM 발생
# 섹션 7 재실행 전 반드시 메모리를 먼저 해제한다.
import gc, torch

try:
    del model
    print("기존 model 해제")
except NameError:
    pass

try:
    del processor
    print("기존 processor 해제")
except NameError:
    pass

try:
    del trainer
    print("기존 trainer 해제")
except NameError:
    pass

gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()

free  = torch.cuda.mem_get_info()[0] / 1024**3
total = torch.cuda.mem_get_info()[1] / 1024**3
print(f"VRAM 여유: {free:.1f}GB / {total:.1f}GB")

# 여유 VRAM 이 15GB 미만이면 런타임을 재시작해야 한다.
# 메뉴: 런타임 -> 세션 다시 시작 후 섹션 2 부터 재실행
assert free >= 15, (
    f"VRAM 여유가 {free:.1f}GB 로 부족합니다. "
    "런타임을 재시작하세요: 메뉴 -> 런타임 -> 세션 다시 시작"
)
print("메모리 확인 완료 - 이후 코드를 실행하세요.")

VRAM 여유: 78.8GB / 79.3GB
메모리 확인 완료 - 이후 코드를 실행하세요.


In [8]:
import gc, torch
from transformers import AutoProcessor, AutoModelForImageTextToText
from peft import LoraConfig, get_peft_model

# 1. GPU 메모리 정보 출력 확인
total_vram  = torch.cuda.get_device_properties(0).total_memory / 1024**3
max_gpu_mem = f"{int(total_vram - 4)}GiB" # 시스템 안전 마진 4GB 확보
print(f"VRAM: {total_vram:.1f}GB  |  max_memory: {max_gpu_mem}")

# 2. 프로세서 및 순수 bfloat16 모델 로드 (BitsAndBytes 양자화 제외)
processor = AutoProcessor.from_pretrained(
    "google/gemma-4-E4B-it",
    trust_remote_code=True,
)
model = AutoModelForImageTextToText.from_pretrained(
    "google/gemma-4-E4B-it",
    device_map="auto",
    max_memory={0: max_gpu_mem, "cpu": "10GiB"},
    trust_remote_code=True,
    dtype=torch.bfloat16, # 'torch_dtype' 경고를 해결하기 위해 최신 스펙인 'dtype' 사용
)
tokenizer = processor.tokenizer
tokenizer.padding_side = "right"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model.config.use_cache = False

# 3. 그라디언트 체크포인팅 활성화
model.gradient_checkpointing_enable(
    gradient_checkpointing_kwargs={"use_reentrant": False}
)

# 🌟 [핵심 수정] 백본 가중치는 얼리고(False), 학습 준비를 마칩니다.
# 이렇게 해야 뒤의 get_peft_model이 LoRA 레이어만 쏙 골라서 True로 켤 수 있습니다.
for param in model.parameters():
    param.requires_grad = False

# 4. LoRA 설정 및 모델 결합
lora_cfg = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_cfg)

# 5. LoRA 레이어만 잘 켜졌는지 파라미터 비중 최종 확인
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"학습 가능 파라미터: {trainable:,}  ({trainable/total*100:.2f}%)")

VRAM: 79.3GB  |  max_memory: 75GiB


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/2076 [00:00<?, ?it/s]

학습 가능 파라미터: 4,538,368  (0.06%)


## 8. 학습

In [9]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import torch
import os
from trl import SFTTrainer, SFTConfig

total_steps  = (len(train_dataset) // (BATCH_SIZE * GRAD_ACCUM)) * NUM_EPOCHS
warmup_steps = max(1, int(total_steps * WARMUP_RATIO))
print(f"전체 스텝: {total_steps}  |  warmup: {warmup_steps}")

tokenizer.model_max_length = MAX_SEQ_LEN

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_steps=warmup_steps,

    # 하드웨어 가속을 위한 핵심 세팅
    fp16=False,
    bf16=True,

    logging_steps=10,
    eval_strategy="steps",
    eval_steps=SAVE_STEPS,
    save_strategy="steps",
    save_steps=SAVE_STEPS,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="none",
    optim="adamw_torch",
    remove_unused_columns=False,
    dataset_text_field="text",
    packing=False,

    # [추가] 코랩 환경 데이터 처리량 향상
    dataloader_num_workers=4,
    dataloader_pin_memory=True
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
    args=training_args,
)

# -------------------------------------------------------
# [수정] bf16 연산에서 절대 터지지 않는 입력 주입 안전 패치
# -------------------------------------------------------
_original_compute_loss = trainer.compute_loss

def patched_compute_loss(model, inputs, return_outputs=False, **kwargs):
    # 입력 데이터가 할당된 GPU 디바이스 자동 추출
    device = inputs["input_ids"].device

    # 정밀도 충돌을 방지하기 위해 정수형(long) 타입과 현재 GPU 디바이스를 명시
    if "token_type_ids" not in inputs:
        inputs["token_type_ids"] = torch.zeros_like(inputs["input_ids"], dtype=torch.long, device=device)

    if "mm_token_type_ids" not in inputs:
        inputs["mm_token_type_ids"] = torch.zeros_like(inputs["input_ids"], dtype=torch.long, device=device)

    return _original_compute_loss(model, inputs, return_outputs=return_outputs, **kwargs)

trainer.compute_loss = patched_compute_loss

# 학습 시작
trainer.train()

adapter_path = os.path.join(OUTPUT_DIR, "lora_adapter_v2")
model.save_pretrained(adapter_path)
tokenizer.save_pretrained(adapter_path)
print(f"어댑터 저장 완료: {adapter_path}")

전체 스텝: 255  |  warmup: 12


Adding EOS to train dataset:   0%|          | 0/5490 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/5490 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/610 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/610 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1, 'bos_token_id': 2, 'pad_token_id': 0}.


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
100,1.358813,1.369161,1.345100,4927068.000000,0.701434
200,1.298924,1.310900,1.287451,9854187.000000,0.709437
258,1.296202,1.307430,1.283282,12710610.000000,0.710024


어댑터 저장 완료: /content/drive/MyDrive/data/RFP_project/peft_output/gemma4-E4B_v2/lora_adapter_v2


## 9. 학습 곡선

In [10]:
log = trainer.state.log_history

train_steps  = [x["step"]      for x in log if "loss"      in x]
train_losses = [x["loss"]      for x in log if "loss"      in x]
eval_steps   = [x["step"]      for x in log if "eval_loss" in x]
eval_losses  = [x["eval_loss"] for x in log if "eval_loss" in x]

plt.figure(figsize=(10, 5))
plt.plot(train_steps, train_losses, label="train loss", alpha=0.7)
if eval_losses:
    plt.plot(eval_steps, eval_losses, label="eval loss", marker="o")
plt.title("Loss Curve")
plt.xlabel("step")
plt.ylabel("loss")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "loss_curve.png"), dpi=150)
plt.show()

best = min(eval_losses) if eval_losses else None
print(f"최저 eval loss : {best:.4f}" if best else "eval loss 없음")


최저 eval loss : 1.3074


## 10. 추론 테스트

In [11]:
import torch

model.eval()

# eval 타입별 1개씩 추론
test_samples = {}
for _, row in eval_df.iterrows():
    t = row["type"]
    if t not in test_samples:
        test_samples[t] = row
    if len(test_samples) >= 5:
        break

for q_type, row in test_samples.items():
    gt_docs  = row["ground_truth_docs"] if isinstance(row["ground_truth_docs"], list) else []
    context  = retrieve_context(gt_docs, max_chunks=2)
    instr    = (
        f"[참고 문서]\n{context}\n\n[질문]\n{row['question']}"
        if context else row["question"]
    )
    full_text   = fmt({"instruction": instr, "response": ""})["text"]
    prompt_text = (
        full_text[:full_text.rfind(RESPONSE_MARKER) + len(RESPONSE_MARKER)]
        if RESPONSE_MARKER in full_text else full_text
    )

    inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    pred = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True,
    ).strip()

    print(f"\n{'='*60}")
    print(f"[Type {q_type} | {row['difficulty']}]")
    print(f"질문 : {row['question']}")
    print(f"정답 : {row['ground_truth_answer']}")
    print(f"예측 : {pred}")



[Type A | 하]
질문 : 한국가스공사의 '차세대 통합정보시스템(ERP) 구축' 사업 예산 규모는 얼마입니까?
정답 : 해당 사업의 예산 규모는 14,107,009,000원입니다.
예측 : 약 1,100억 원입니다.<end_of_turn>

[Type B | 하]
질문 : 한국가스공사의 '차세대 ERP 구축' 사업과 고려대학교의 '차세대 포털·학사 정보시스템 구축사업' 간의 예산 규모 차이는 얼마입니까?
정답 : 한국가스공사 사업 예산(14,107,009,000원)이 고려대학교 사업 예산(11,270,000,000원)보다 2,837,009,000원 더 큽니다.
예측 : 한국가스공사의 '차세대 ERP 구축' 사업은 1,100억 원이며, 고려대학교의 '차세대 포털·학사 정보시스템 구축사업'은 1,100억 원입니다. 따라서 두 사업 간의 예산 규모 차이는 없습니다.<end_of_turn>

[Type C | 중]
질문 : 앞서 알려주신 신규 도입 항목들의 용도에 맞게, 해당 문서에서 언급하는 제조사 기술지원 종료 시점이 언제인지 말씀해 주세요.
정답 : 한국가스공사의 차세대 ERP 시스템 구축 문서에서 제조사(SAP)의 기술지원 종료 시점은 2027년으로 명시되어 있습니다.
예측 : 제조사 기술지원 종료 시점은 2027년입니다.<end_of_turn>

[Type D | 하]
질문 : 고려대학교의 '차세대 포털·학사 정보시스템 구축사업'을 수주하기 위해서는 총괄 책임자가 반드시 정보보안기사 및 정보처리기사 자격증 원본을 필수로 함께 제출해야 합니까?
정답 : 해당 문서에는 총괄 담당자의 정보보안기사 또는 정보처리기사 자격증 필수 제출이나 요구 여부에 대한 명시적인 지침 또는 제한 정보가 포함되어 있지 않습니다.
예측 : 네, 총괄 책임자는 정보보안기사 및 정보처리기사 자격증 원본을 필수로 함께 제출해야 합니다.<end_of_turn>

[Type E | 하]
질문 : 코이까(KOICA) 전쟈죠달의 국회 뱡송시쓰템 구축 사업의 총 예싼은 얼마로 잡혀 있습니가?

## 11. 어댑터 다운로드

In [12]:
# 드라이브에 이미 저장됐으므로 로컬 다운로드가 필요하면 아래 주석 해제
# import shutil
# from google.colab import files
# shutil.make_archive("/content/lora_adapter", "zip", adapter_path)
# files.download("/content/lora_adapter.zip")
print(f"어댑터 위치: {adapter_path}")
print("로컬 이전: rclone 또는 구글 드라이브 웹에서 직접 다운로드")


어댑터 위치: /content/drive/MyDrive/data/RFP_project/peft_output/gemma4-E4B_v2/lora_adapter_v2
로컬 이전: rclone 또는 구글 드라이브 웹에서 직접 다운로드
